# Experimento 5B: PubMedBERT + augmentation con PAREJAS DE ENTIDAD NUEVAS para ALTERNATIVE_NAME

**Objetivo:** en 5A, parafrasear las mismas 95 parejas de `ALTERNATIVE_NAME`
no ayudo (incluso empeoro un poco, 0.131->0.111 en ciego calibrado) -- el
diagnostico (ver `DATA-AUGMENTATION-RESUMEN-COMPLETO.md`) fue que el 74.5%
de los casos reales se siguen prediciendo como `no_relation`, y parafrasear
las MISMAS parejas no le da al modelo entidades nuevas con las que
generalizar el patron.

Este experimento anade 20 parejas de entidad **nuevas** (no las 95/97 de
siempre), verificadas de antemano como vocabulario medico estandar (pares
anatomicos sustantivo<->adjetivo: kidney/renal, liver/hepatic, brain/cerebral...
-- ver `baseline/augment_llm_newpairs.py`), no inventadas por el LLM -- el
LLM solo escribio la frase alrededor de cada pareja ya verificada.

**Nota:** son 20 de las 47 parejas planeadas (la cuota diaria gratuita de
Gemini se agoto a mitad de la generacion) -- esto es una señal temprana con
menos de la mitad del refuerzo previsto, no el experimento completo. Cuando
se generen las 27 restantes se puede repetir con el set completo.

**Datos:** `eng_train_plus_llmaug_v2.txt` = `eng_train.txt` (12739) +
`eng_train_llmaug.txt` (318, parafraseos de 5A) + `eng_train_llmaug_newpairs.txt`
(20, parejas nuevas). Comparacion a tres bandas: 3A (sin augment) / 5A (solo
parafraseo) / 5B (parafraseo + parejas nuevas) -- para aislar el efecto
marginal de las parejas nuevas frente al parafraseo solo.

## 1. Setup

In [ ]:
import os
HF_CACHE_DIR = os.path.expanduser("~/hf_cache")
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE_DIR, "hub")
os.makedirs(os.environ["HF_HUB_CACHE"], exist_ok=True)
import huggingface_hub.constants as hfc
assert hfc.HF_HUB_CACHE == os.environ["HF_HUB_CACHE"], (
    "Reinicia el kernel y ejecuta esta celda ANTES de cualquier import de HF/opennre.")
print("HF cache:", os.environ["HF_HUB_CACHE"])

In [ ]:
import json, time, logging, gc, random
from collections import Counter
from pathlib import Path

import nltk, pandas as pd, torch, numpy as np

logging.getLogger("transformers").setLevel(logging.ERROR)
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
import opennre

try:
    nltk.data.find("tokenizers/punkt_tab")
except LookupError:
    nltk.download("punkt_tab", quiet=True)

import sys
sys.path.insert(0, "../baseline")
from patch_opennre import add_macro_f1_metric, fix_entity_markers
add_macro_f1_metric()
fix_entity_markers()   # identico a 3A/4B/4C/5A
from score import evaluate

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2. Preparar el train combinado (originales + parafraseos + parejas nuevas)

In [ ]:
DATA_DIR         = Path("../data/english")
ORIG_TRAIN       = DATA_DIR / "eng_train.txt"
LLMAUG_PARAPHRASE= DATA_DIR / "eng_train_llmaug.txt"
LLMAUG_NEWPAIRS  = DATA_DIR / "eng_train_llmaug_newpairs.txt"
COMBINED_TRAIN   = DATA_DIR / "eng_train_plus_llmaug_v2.txt"

for p in (ORIG_TRAIN, LLMAUG_PARAPHRASE, LLMAUG_NEWPAIRS):
    assert p.exists(), f"FALTA {p}"

orig_lines  = [l for l in open(ORIG_TRAIN, encoding="utf-8") if l.strip()]
para_lines  = [l for l in open(LLMAUG_PARAPHRASE, encoding="utf-8") if l.strip()]
new_lines   = [l for l in open(LLMAUG_NEWPAIRS, encoding="utf-8") if l.strip()]
with open(COMBINED_TRAIN, "w", encoding="utf-8") as f:
    f.writelines(orig_lines); f.writelines(para_lines); f.writelines(new_lines)

print(f"original: {len(orig_lines)} | parafraseos (5A): {len(para_lines)} | "
      f"parejas nuevas: {len(new_lines)} | combinado: {len(orig_lines)+len(para_lines)+len(new_lines)}")

alt_orig = sum(1 for l in orig_lines if json.loads(l)["relation"] == "ALTERNATIVE_NAME")
alt_para = sum(1 for l in para_lines if json.loads(l)["relation"] == "ALTERNATIVE_NAME")
alt_new  = len(new_lines)  # todas son ALTERNATIVE_NAME
print(f"\nALTERNATIVE_NAME: {alt_orig} original + {alt_para} parafraseado (parejas repetidas) "
      f"+ {alt_new} parejas NUEVAS = {alt_orig+alt_para+alt_new} total")
print(f"Parejas de entidad UNICAS para ALTERNATIVE_NAME: {alt_orig} (las de siempre) + {alt_new} (nuevas) "
      f"-- los {alt_para} parafraseos NO anaden parejas nuevas, solo mas frases de las mismas")

## 3. Configuracion -- identica a 3A/5A salvo el fichero de train

In [ ]:
MODEL_NAME      = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
EXPERIMENT_NAME = "pubmedbert_llmaug_v2"
TECHNIQUE       = ("eng_train.txt + eng_train_llmaug.txt (318, parafraseos) + "
                    "eng_train_llmaug_newpairs.txt (20, parejas nuevas verificadas de ALTERNATIVE_NAME) "
                    "-- sobre fix_entity_markers(), unico cambio vs 3A/5A")

MAX_LENGTH     = 256
BATCH_SIZE     = 16
LEARNING_RATE  = 2e-5
EPOCHS         = 15
WARMUP_STEPS   = 300
SEED           = 42
GRAD_CLIP_NORM = 1.0

TRAIN_DATA  = COMBINED_TRAIN
DEV_DATA    = DATA_DIR / "eng_dev.txt"
REL2ID_PATH = DATA_DIR / "rel2id.json"
for p in (TRAIN_DATA, DEV_DATA, REL2ID_PATH):
    assert p.exists(), f"FALTA {p}"

with open(REL2ID_PATH) as f:
    rel2id = json.load(f)
id2rel = {v: k for k, v in rel2id.items()}
NO_REL_ID = rel2id["no_relation"]
print(f"Clases: {len(rel2id)} | modelo: {MODEL_NAME}")

OUT_DIR = Path(f"../outputs/5B-pubmedbert-llmaug-newpairs/seed{SEED}")
OUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = OUT_DIR / f"eng_{EXPERIMENT_NAME}.pth.tar"
print("Salida:", OUT_DIR)

## 4. Entrenamiento -- misma funcion que 1G/3A/4B/4C/5A

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


from opennre.framework.utils import AverageMeter
from tqdm import tqdm

def train_with_history(fw, max_epoch, metric="macro_f1"):
    history, best_metric = [], 0
    for epoch in range(max_epoch):
        fw.train()
        avg_loss, avg_acc = AverageMeter(), AverageMeter()
        t = tqdm(fw.train_loader, desc=f"Epoch {epoch}")
        for data in t:
            if torch.cuda.is_available():
                for i in range(len(data)):
                    try: data[i] = data[i].cuda()
                    except Exception: pass
            label, args = data[0], data[1:]
            logits = fw.parallel_model(*args)
            loss = fw.criterion(logits, label)
            _, pred = logits.max(-1)
            acc = float((pred == label).long().sum()) / label.size(0)
            avg_loss.update(loss.item(), 1); avg_acc.update(acc, 1)
            t.set_postfix(loss=avg_loss.avg, acc=avg_acc.avg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(fw.model.parameters(), GRAD_CLIP_NORM)
            fw.optimizer.step()
            if fw.scheduler is not None: fw.scheduler.step()
            fw.optimizer.zero_grad()
        val = fw.eval_model(fw.val_loader)
        rec = {"epoch": epoch, "train_loss": avg_loss.avg, "train_acc": avg_acc.avg,
               "val_acc": val["acc"], "val_micro_p": val["micro_p"], "val_micro_r": val["micro_r"],
               "val_micro_f1": val["micro_f1"], "val_macro_f1": val["macro_f1"]}
        history.append(rec)
        print(f"Epoch {epoch}: loss={rec['train_loss']:.4f} "
              f"val_micro_f1={rec['val_micro_f1']:.4f} val_macro_f1={rec['val_macro_f1']:.4f}")
        if val[metric] > best_metric:
            print(f"  -> nuevo mejor {metric}={val[metric]:.4f}, guardando checkpoint")
            folder = "/".join(fw.ckpt.split("/")[:-1])
            if folder and not os.path.exists(folder): os.makedirs(folder, exist_ok=True)
            torch.save({"state_dict": fw.model.state_dict()}, fw.ckpt)
            best_metric = val[metric]
    print(f"Mejor {metric} en val: {best_metric:.4f}")
    return history

In [ ]:
set_seed(SEED)

encoder = opennre.encoder.BERTEntityEncoder(max_length=MAX_LENGTH, pretrain_path=MODEL_NAME)
model = opennre.model.SoftmaxNN(sentence_encoder=encoder, num_class=len(rel2id), rel2id=rel2id)
framework = opennre.framework.SentenceRE(
    model=model, train_path=str(TRAIN_DATA), val_path=str(DEV_DATA), test_path=str(DEV_DATA),
    ckpt=str(CKPT_PATH), batch_size=BATCH_SIZE, max_epoch=EPOCHS, lr=LEARNING_RATE,
    opt="adamw", warmup_step=WARMUP_STEPS)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parametros: {n_params:,}")
print(f"Instancias de train: {len(orig_lines) + len(para_lines) + len(new_lines)}")

t0 = time.time()
history = train_with_history(framework, EPOCHS, metric="macro_f1")
train_minutes = (time.time() - t0) / 60
with open(OUT_DIR / f"history_{EXPERIMENT_NAME}.json", "w") as f:
    json.dump(history, f, indent=2)
best = max(history, key=lambda h: h["val_macro_f1"])
macro_f1_curado = best["val_macro_f1"]
print(f"\nEntreno: {train_minutes:.1f} min | mejor epoch={best['epoch']} macro_f1_curado(dev)={macro_f1_curado:.4f}")

## 5. Inferencia en blind + evaluacion oficial (argmax)

In [ ]:
BLIND_DEV_PATH = DATA_DIR / "eng_dev_blind.txt"
BLIND_GOLD_TSV = DATA_DIR / "eng-dev-rel.tsv"
for p in (BLIND_DEV_PATH, BLIND_GOLD_TSV):
    assert p.exists(), f"FALTA {p}"

blind_raw = [json.loads(l) for l in open(BLIND_DEV_PATH, encoding="utf-8") if l.strip()]
gold_df_blind = pd.read_csv(BLIND_GOLD_TSV, sep="\t")
print(f"candidatos blind: {len(blind_raw)} | gold real: {len(gold_df_blind)}")

model.load_state_dict(torch.load(str(CKPT_PATH), map_location="cpu")["state_dict"])
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = model.to(device); model.eval()

t0 = time.time()
all_probs = np.zeros((len(blind_raw), len(rel2id)), dtype=np.float32)
with torch.no_grad():
    for s in range(0, len(blind_raw), 64):
        batch = blind_raw[s:s + 64]
        tok = [encoder.tokenize({"text": i["text"], "h": {"pos": i["h"]["pos"]},
                                 "t": {"pos": i["t"]["pos"]}}) for i in batch]
        fields = [torch.cat([t[k] for t in tok], dim=0).to(device) for k in range(len(tok[0]))]
        logits = model(*fields)
        all_probs[s:s + len(batch)] = torch.softmax(logits, dim=-1).cpu().numpy()
blind_minutes = (time.time() - t0) / 60
np.save(OUT_DIR / f"blind_probs_{EXPERIMENT_NAME}.npy", all_probs)
print(f"Inferencia blind: {blind_minutes:.1f} min")

def rows_from_preds(pred_ids):
    labels = [id2rel[i] for i in pred_ids]
    return [
        {"document_id": inst["doc_id"], "relation": rel,
         "head_text": inst["h"]["name"], "head_span": inst["head_span"], "head_type": inst["head_type"],
         "tail_text": inst["t"]["name"], "tail_span": inst["tail_span"], "tail_type": inst["tail_type"]}
        for inst, rel in zip(blind_raw, labels) if rel != "no_relation"
    ]

argmax_ids = all_probs.argmax(axis=1)
res_argmax = evaluate(pd.DataFrame(rows_from_preds(argmax_ids)), gold_df_blind)
macro_f1_ciego_argmax = res_argmax["macro_f1"]
print(f"Macro F1 ciego (argmax puro): {macro_f1_ciego_argmax:.4f}")

## 6. Calibracion de threshold (grid fino) -- comparacion a tres bandas: 3A / 5A / 5B

In [ ]:
def preds_at_threshold(probs, threshold):
    probs2 = probs.copy()
    probs2[:, NO_REL_ID] = -1
    best_rel_id = probs2.argmax(axis=1)
    best_rel_prob = probs[np.arange(len(probs)), best_rel_id]
    return np.where(best_rel_prob >= threshold, best_rel_id, NO_REL_ID)

def eval_at(probs, threshold):
    pred_ids = preds_at_threshold(probs, threshold)
    return evaluate(pd.DataFrame(rows_from_preds(pred_ids)), gold_df_blind)

FINE_GRID = [round(float(x), 4) for x in np.arange(0.85, 0.9901, 0.001)] + \
            [round(float(x), 4) for x in np.arange(0.990, 0.9991, 0.001)] + \
            [0.9995, 0.9999]
FINE_GRID = sorted(set(FINE_GRID))

sweep = [(th, eval_at(all_probs, th)["macro_f1"]) for th in FINE_GRID]
best_threshold, macro_f1_ciego_calibrado = max(sweep, key=lambda x: x[1])
en_borde = best_threshold == FINE_GRID[-1]
per_relation_calibrado = eval_at(all_probs, best_threshold)["per_relation"]
print(f"Mejor threshold (fino): {best_threshold:.4f} -> Macro F1 ciego calibrado = {macro_f1_ciego_calibrado:.4f}"
      f"{'  [BORDE DEL GRID -- revisar]' if en_borde else ''}")

baseline_3a = json.load(open("../outputs/3A-pubmedbert-fixed/seed42/results_seed_summary.json"))
baseline_5a = json.load(open("../outputs/5A-pubmedbert-llmaug/seed42/results_seed_summary.json"))

print(f"\n{'':<32}{'argmax':>10}{'calibrado':>12}{'threshold':>12}")
print(f"{'3A (sin augment)':<32}{baseline_3a['macro_f1_ciego_argmax']:>10.4f}"
      f"{baseline_3a['macro_f1_ciego_calibrado']:>12.4f}{baseline_3a['best_threshold_fino']:>12.4f}")
print(f"{'5A (solo parafraseo)':<32}{baseline_5a['macro_f1_ciego_argmax']:>10.4f}"
      f"{baseline_5a['macro_f1_ciego_calibrado']:>12.4f}{baseline_5a['best_threshold_fino']:>12.4f}")
print(f"{'5B (parafraseo + parejas nuevas)':<32}{macro_f1_ciego_argmax:>10.4f}"
      f"{macro_f1_ciego_calibrado:>12.4f}{best_threshold:>12.4f}")
print(f"\ndelta 5B vs 3A: {macro_f1_ciego_calibrado - baseline_3a['macro_f1_ciego_calibrado']:+.4f}")
print(f"delta 5B vs 5A: {macro_f1_ciego_calibrado - baseline_5a['macro_f1_ciego_calibrado']:+.4f}  "
      f"(este es el efecto marginal de las 20 parejas nuevas)")

## 7. Foco en ALTERNATIVE_NAME -- la relacion que motivo este experimento

In [ ]:
alt_5b = per_relation_calibrado.get("ALTERNATIVE_NAME", {})
alt_5a = baseline_5a.get("per_relation_ciego_calibrado", {}).get("ALTERNATIVE_NAME", {})
print(f"{'':<12}{'F1':>8}{'Precision':>12}{'Recall':>10}")
print(f"{'5A (solo parafraseo)':<22}{alt_5a.get('f1',float('nan')):>8.3f}"
      f"{alt_5a.get('precision',float('nan')):>12.3f}{alt_5a.get('recall',float('nan')):>10.3f}")
print(f"{'5B (+ parejas nuevas)':<22}{alt_5b.get('f1',float('nan')):>8.3f}"
      f"{alt_5b.get('precision',float('nan')):>12.3f}{alt_5b.get('recall',float('nan')):>10.3f}")
print("\n(recordatorio: solo 20 de las 47 parejas nuevas planeadas -- señal parcial, no el experimento completo)")

# matriz de confusion real de ALTERNATIVE_NAME en este run
from collections import Counter
pred_ids_best = preds_at_threshold(all_probs, best_threshold)
pred_labels_best = [id2rel[i] for i in pred_ids_best]
def _key(doc, hs, ts): return f"{doc}|{hs}|{ts}"
gold_rel = {_key(r.document_id, r.head_span, r.tail_span): r.relation for r in gold_df_blind.itertuples()}
confusion = Counter()
total_alt_gold = 0
for inst, pred in zip(blind_raw, pred_labels_best):
    k = _key(inst["doc_id"], inst["head_span"], inst["tail_span"])
    if gold_rel.get(k) == "ALTERNATIVE_NAME":
        total_alt_gold += 1
        confusion[pred] += 1
print(f"\nMatriz de confusion real ALTERNATIVE_NAME (gold={total_alt_gold}):")
for pred, c in confusion.most_common(10):
    print(f"  predicho como {pred}: {c} ({100*c/total_alt_gold:.1f}%)")

## 8. Guardar resultados

In [ ]:
results = {
    "exp": EXPERIMENT_NAME,
    "model": MODEL_NAME,
    "technique": TECHNIQUE,
    "seed": SEED,
    "hyperparameters": {
        "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE,
        "epochs": EPOCHS, "warmup_steps": WARMUP_STEPS, "neg_ratio": 3, "seed": SEED,
    },
    "n_train_original": len(orig_lines),
    "n_train_llmaug_paraphrase": len(para_lines),
    "n_train_llmaug_newpairs": len(new_lines),
    "n_train_total": len(orig_lines) + len(para_lines) + len(new_lines),
    "newpairs_completitud": "20/47 -- cuota diaria agotada a mitad de generacion",
    "macro_f1_curado": macro_f1_curado,
    "macro_f1_ciego_argmax": macro_f1_ciego_argmax,
    "best_threshold_fino": best_threshold,
    "macro_f1_ciego_calibrado": macro_f1_ciego_calibrado,
    "en_borde_del_grid_fino": en_borde,
    "per_relation_ciego_calibrado": per_relation_calibrado,
    "train_minutes": round(train_minutes, 1),
    "blind_minutes": round(blind_minutes, 1),
    "comparison": {
        "3A_sin_augment": baseline_3a["macro_f1_ciego_calibrado"],
        "5A_solo_parafraseo": baseline_5a["macro_f1_ciego_calibrado"],
        "5B_parafraseo_mas_parejas_nuevas": macro_f1_ciego_calibrado,
        "delta_vs_3A": macro_f1_ciego_calibrado - baseline_3a["macro_f1_ciego_calibrado"],
        "delta_vs_5A_efecto_marginal_parejas_nuevas": macro_f1_ciego_calibrado - baseline_5a["macro_f1_ciego_calibrado"],
    },
}
with open(OUT_DIR / "results_seed_summary.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Guardado:", OUT_DIR / "results_seed_summary.json")
print(f"\n3A: {baseline_3a['macro_f1_ciego_calibrado']:.4f}  ->  5A: {baseline_5a['macro_f1_ciego_calibrado']:.4f}  "
      f"->  5B: {macro_f1_ciego_calibrado:.4f}")

## 9. Conclusion (rellenar tras ver los resultados)

Recordatorios antes de sacar conclusiones:
- **Una sola seed** -- std medida entre seeds para PubMedBERT en ciego calibrado: ~0.009-0.017.
- **Solo 20/47 parejas nuevas** -- si el efecto es positivo pero pequeño, podria
  crecer al completar las 27 restantes; si es negativo o nulo, ya es una señal
  util para no invertir mas en esta via sin repensarla.